# Climate TRACE ownership, first look

This notebook looks at who owns the facilities in the Climate TRACE emissions data. I want to know how much of the emissions we care about can be linked to a company, how ownership is recorded, and what traps there are before I design the attribution logic.

The input is the ownership package from release v5.11.0, downloaded on 30 September 2026 for all US sources in every sector. It is a single 5.6 MB CSV with 20,210 rows, so I read it directly instead of converting it to Parquet. I also use the emissions Parquet file built in notebook 01.

In [14]:
import sys
from pathlib import Path
import duckdb

OWN = "../../data/climate_trace/v5.11.0/ownership/DATA/ownership_v5_11_0.csv"
EM = "../../data/climate_trace/v5.11.0/emissions/emissions_co2e_100yr_v5_11_0.parquet"

assert Path(OWN).exists(), f"Missing ownership CSV: {Path(OWN).resolve()}"
assert Path(EM).exists(), f"Missing emissions Parquet, run notebook 01 first: {Path(EM).resolve()}"
print(sys.executable, sys.version.split()[0], duckdb.__version__)

/opt/miniconda3/envs/carboncap/bin/python 3.12.14 1.5.6


## 1. Views

`own` is the ownership file kept as text, with the two percentage columns cast to numbers. `facility` has one row per facility with its total emissions for 2021 to 2025. Area sources are left out because they have no owners.

In [15]:
con = duckdb.connect()

# Ownership kept as text like bronze, with the two percentages cast for analysis
con.sql(f"""
    CREATE OR REPLACE VIEW own AS
    SELECT
        *,
        TRY_CAST(overall_share_percent AS DOUBLE)   AS share_pct,
        TRY_CAST(percentage_of_operation AS DOUBLE) AS operation_pct
    FROM read_csv('{OWN}', all_varchar=true)
""")

# One row per facility with its total emissions for 2021 to 2025
con.sql(f"""
    CREATE OR REPLACE VIEW facility AS
    SELECT
        source_id,
        ANY_VALUE(source_name)                       AS source_name,
        ANY_VALUE(subsector)                         AS subsector,
        SUM(TRY_CAST(emissions_quantity AS DOUBLE))  AS emissions_t
    FROM '{EM}'
    WHERE geometry_ref IS NULL
    GROUP BY source_id
""")

## 2. Shape of the file

In [16]:
con.sql("""
    SELECT
        COUNT(*)                          AS row_count,
        COUNT(DISTINCT source_id)         AS sources,
        COUNT(DISTINCT parent_entity_id)  AS parents,
        COUNT(DISTINCT source_subsector)  AS subsectors,
        COUNT(*) FILTER (WHERE overall_share_percent IS NOT NULL AND share_pct IS NULL)       AS failed_share_casts,
        COUNT(*) FILTER (WHERE percentage_of_operation IS NOT NULL AND operation_pct IS NULL) AS failed_operation_casts
    FROM own
""").show()

┌───────────┬─────────┬─────────┬────────────┬────────────────────┬────────────────────────┐
│ row_count │ sources │ parents │ subsectors │ failed_share_casts │ failed_operation_casts │
│   int64   │  int64  │  int64  │   int64    │       int64        │         int64          │
├───────────┼─────────┼─────────┼────────────┼────────────────────┼────────────────────────┤
│     20210 │    6564 │    2884 │         14 │                  0 │                      0 │
└───────────┴─────────┴─────────┴────────────┴────────────────────┴────────────────────────┘



In [17]:
# Test: ownership rows whose source_id is not in the emissions data. No rows means the test passed.
con.sql("""
    SELECT DISTINCT source_id, source_name, source_subsector
    FROM own
    ANTI JOIN facility USING (source_id)
""").show()

┌───────────┬─────────────┬──────────────────┐
│ source_id │ source_name │ source_subsector │
│  varchar  │   varchar   │     varchar      │
└───────────┴─────────────┴──────────────────┘
                    0 rows                  



The file has 20,210 rows covering 6,564 sources and 2,884 different top level owners, across 14 subsectors. Both percentage columns convert cleanly to numbers.

The test returned nothing, so every ownership row joins to a facility in the emissions data.

## 3. How much of the emissions have an owner

In [18]:
con.sql("""
    WITH owned AS (SELECT DISTINCT source_id FROM own)
    SELECT
        f.subsector,
        COUNT(*)                                          AS sources,
        COUNT(o.source_id)                                AS sources_with_owner,
        ROUND(100.0 * COUNT(o.source_id) / COUNT(*), 1)   AS pct_sources,
        ROUND(SUM(f.emissions_t) / 1e6, 1)                AS emissions_mt,
        ROUND(100 * COALESCE(SUM(f.emissions_t) FILTER (WHERE o.source_id IS NOT NULL), 0)
              / SUM(f.emissions_t), 1)                     AS pct_emissions
    FROM facility f
    LEFT JOIN owned o USING (source_id)
    GROUP BY f.subsector
    ORDER BY emissions_mt DESC
""").show(max_rows=40)

┌───────────────────────────────────────────────┬─────────┬────────────────────┬─────────────┬──────────────┬───────────────┐
│                   subsector                   │ sources │ sources_with_owner │ pct_sources │ emissions_mt │ pct_emissions │
│                    varchar                    │  int64  │       int64        │   double    │    double    │    double     │
├───────────────────────────────────────────────┼─────────┼────────────────────┼─────────────┼──────────────┼───────────────┤
│ electricity-generation                        │    2528 │               2383 │        94.3 │       8017.5 │          99.7 │
│ oil-and-gas-production                        │     572 │                  0 │         0.0 │       2399.3 │           0.0 │
│ oil-and-gas-refining                          │     134 │                134 │       100.0 │        923.4 │         100.0 │
│ coal-mining                                   │     692 │                556 │        80.3 │        849.9 │         

In [19]:
# The same coverage for the sectors in scope, as one number
con.sql("""
    WITH owned AS (SELECT DISTINCT source_id FROM own)
    SELECT
        ROUND(SUM(f.emissions_t) / 1e6, 1) AS in_scope_mt,
        ROUND(100 * COALESCE(SUM(f.emissions_t) FILTER (WHERE o.source_id IS NOT NULL), 0)
              / SUM(f.emissions_t), 1)      AS pct_with_owner
    FROM facility f
    LEFT JOIN owned o USING (source_id)
    WHERE f.subsector IN (
        'electricity-generation', 'oil-and-gas-production', 'oil-and-gas-refining',
        'oil-and-gas-transport', 'coal-mining', 'chemicals', 'other-chemicals',
        'petrochemical-steam-cracking', 'iron-and-steel', 'other-metals', 'aluminum',
        'cement', 'lime', 'glass', 'copper-mining', 'iron-mining'
    )
""").show()

┌─────────────┬────────────────┐
│ in_scope_mt │ pct_with_owner │
│   double    │     double     │
├─────────────┼────────────────┤
│     14626.8 │           72.9 │
└─────────────┴────────────────┘



* Where ownership exists, coverage is excellent. Power plants with an owner account for 99.7% of power emissions, and refining, petrochemicals, cement, steel, copper and iron mining are all close to 100%. Coal mining is at 92% and chemicals at 69%.
* Oil and gas production and oil and gas transport have no ownership data at all. Together they are about 3,090 Mt, the second largest block of emissions in our sectors after power. That fits the earlier suspicion that these sources are basins rather than facilities.
* Lime, glass, other chemicals and other metals also have no owners, which takes them out of scope in practice.
* Across the sectors in scope, 72.9% of the 14,627 Mt can be linked to at least one owner. Nearly all of the gap is oil and gas production and transport.

## 4. Identifiers

In [20]:
con.sql("""
    SELECT
        COALESCE(parent_entity_type, '<empty>')            AS parent_type,
        COUNT(*)                                            AS row_count,
        COUNT(DISTINCT parent_entity_id)                    AS parents,
        ROUND(100.0 * COUNT(parent_lei)    / COUNT(*), 1)   AS pct_with_lei,
        ROUND(100.0 * COUNT(parent_permid) / COUNT(*), 1)   AS pct_with_permid
    FROM own
    GROUP BY 1
    ORDER BY row_count DESC
""").show()

┌────────────────┬───────────┬─────────┬──────────────┬─────────────────┐
│  parent_type   │ row_count │ parents │ pct_with_lei │ pct_with_permid │
│    varchar     │   int64   │  int64  │    double    │     double      │
├────────────────┼───────────┼─────────┼──────────────┼─────────────────┤
│ legal entity   │     10820 │    1444 │         97.2 │            97.6 │
│ <empty>        │      6205 │    1080 │          0.0 │             0.0 │
│ unknown entity │      1398 │       5 │         99.9 │            99.9 │
│ person         │      1043 │       2 │        100.0 │           100.0 │
│ state          │       395 │     173 │         84.1 │            98.2 │
│ state body     │       344 │     175 │         95.6 │            99.1 │
│ arrangement    │         5 │       5 │         80.0 │            80.0 │
└────────────────┴───────────┴─────────┴──────────────┴─────────────────┘



* About 97% of rows for legal entities carry both an LEI and a PermID. That means I can match owners to SEC filers on identifiers instead of names.
* The 6,205 rows with no owner type are aviation and cattle operations. The aviation rows have no owner at all, they list operators instead (section 8). The cattle owners are small private farms.
* "small shareholder(s)" and "member/employee owned" are placeholder owners, not real companies.

## 5. How long the chains are

In [21]:
# Number of links between the top owner and the facility
con.sql("""
    SELECT
        len(string_split(ownership_path, ' -> ')) - 1  AS links,
        COUNT(*)                                       AS row_count
    FROM own
    WHERE ownership_path IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").show()

┌───────┬───────────┐
│ links │ row_count │
│ int64 │   int64   │
├───────┼───────────┤
│     1 │      4504 │
│     2 │      2839 │
│     3 │      4528 │
│     4 │      2560 │
│     5 │      1123 │
│     6 │       441 │
│     7 │       192 │
│     8 │       159 │
│     9 │        73 │
│    10 │        27 │
│    11 │        10 │
│    12 │         6 │
│    13 │         4 │
└───────┴───────────┘
       13 rows     



Most chains have between one and four links between the top owner and the facility, but some go up to 13. The whole chain is stored as one text column, so I will need to split it to work with the companies in between.

## 6. Do the shares add up to 100?

In [22]:
con.sql("""
    WITH per_source AS (
        SELECT
            source_subsector,
            source_id,
            COUNT(*)          AS paths,
            COUNT(share_pct)  AS paths_with_share,
            SUM(share_pct)    AS total_share
        FROM own
        WHERE ownership_path IS NOT NULL
        GROUP BY ALL
    )
    SELECT
        source_subsector,
        COUNT(*) FILTER (WHERE paths_with_share = 0)                                             AS no_share_known,
        COUNT(*) FILTER (WHERE paths_with_share BETWEEN 1 AND paths - 1)                         AS some_shares_known,
        COUNT(*) FILTER (WHERE paths_with_share = paths AND total_share < 99.5)                  AS below_100,
        COUNT(*) FILTER (WHERE paths_with_share = paths AND total_share BETWEEN 99.5 AND 100.5)  AS about_100,
        COUNT(*) FILTER (WHERE paths_with_share = paths AND total_share > 100.5)                 AS above_100
    FROM per_source
    GROUP BY source_subsector
    ORDER BY source_subsector
""").show()

┌───────────────────────────────────────┬────────────────┬───────────────────┬───────────┬───────────┬───────────┐
│           source_subsector            │ no_share_known │ some_shares_known │ below_100 │ about_100 │ above_100 │
│                varchar                │     int64      │       int64       │   int64   │   int64   │   int64   │
├───────────────────────────────────────┼────────────────┼───────────────────┼───────────┼───────────┼───────────┤
│ aluminum                              │              5 │                 0 │         0 │         0 │         0 │
│ cement                                │             12 │                25 │         0 │         2 │        45 │
│ chemicals                             │             25 │                 0 │         0 │         0 │         0 │
│ coal-mining                           │             97 │               144 │         0 │       254 │        61 │
│ copper-mining                         │             41 │                 0 │  

Not reliably, and for two different reasons.

* For refining, petrochemicals, chemicals, copper mining, iron mining and aluminum, no share is known for any facility, because the last link to the plant is marked as an unknown percentage. Motiva's Port Arthur refinery, for example, sits under Saudi Aramco with an unknown share. I need a rule for this, such as treating a single direct owner as owning 100% and flagging the figure as an estimate.
* Where shares are known, many facilities add up to more than 100. The next section shows why.

## 7. Why shares go above 100

In [23]:
con.sql("""
    SELECT parent_name, parent_entity_type, share_pct, ownership_path
    FROM own
    WHERE source_name = '59th Street'
    ORDER BY share_pct DESC
""").show(max_width=250)

┌─────────────────────────┬────────────────────┬───────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│       parent_name       │ parent_entity_type │ share_pct │                                                                        ownership_path                                                                        │
│         varchar         │      varchar       │  double   │                                                                           varchar                                                                            │
├─────────────────────────┼────────────────────┼───────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Consolidated Edison Inc │ legal entity       │     100.0 │ Consolidated Edison Inc -> Consolidated Edison Company of N

Con Edison's 59th Street plant appears six times. One row has Consolidated Edison Inc as the owner with 100%. The other five go further up the chain to Con Edison's own shareholders: small shareholders with 70.19%, Vanguard 12.13%, BlackRock 10.1%, State Street 7.58%, and even a BlackRock subsidiary that owns part of BlackRock. The file holds several levels of the same chain at once, so adding up every row for a plant counts it more than once.

The next query does the naive thing to show how much this matters: it multiplies each facility's emissions by every row's share and adds the results up by top owner.

In [24]:
# Deliberately wrong: sums every row, so investors above the real owner get emissions too
con.sql("""
    SELECT
        o.parent_name,
        o.parent_entity_type,
        COUNT(DISTINCT o.source_id)                             AS sources,
        ROUND(SUM(f.emissions_t * o.share_pct / 100) / 1e6, 1)  AS naive_attributed_mt
    FROM own o
    JOIN facility f USING (source_id)
    GROUP BY ALL
    ORDER BY naive_attributed_mt DESC NULLS LAST
    LIMIT 15
""").show()

┌─────────────────────────────────────┬────────────────────┬─────────┬─────────────────────┐
│             parent_name             │ parent_entity_type │ sources │ naive_attributed_mt │
│               varchar               │      varchar       │  int64  │       double        │
├─────────────────────────────────────┼────────────────────┼─────────┼─────────────────────┤
│ small shareholder(s)                │ unknown entity     │    1201 │              2253.8 │
│ Duke Energy Corp                    │ legal entity       │      51 │               396.7 │
│ The Vanguard Group Inc              │ legal entity       │    1167 │               390.5 │
│ Southern Co                         │ legal entity       │      38 │               379.7 │
│ BlackRock Inc                       │ legal entity       │    1160 │               338.6 │
│ Vistra Corp                         │ legal entity       │      61 │               334.0 │
│ small shareholder(s)                │ unknown entity     │     136 │

The result is absurd. "small shareholder(s)" comes out as the largest emitter in the country, and Vanguard, BlackRock and State Street all make the top 15 although they don't run a single plant. This is the trap: the top of the chain is often an investor, not the company that actually owns the facility.

The rule I plan to use instead is to pick, for each facility, the row whose owner is a company in our universe, matched on LEI or PermID, and ignore the rows above it. For Con Edison that means the 100% row and nothing else. This needs its own ADR, including what to do when one listed company owns another.

## 8. Aviation operators

In [25]:
con.sql("""
    SELECT source_name, source_operator, operation_pct
    FROM own
    WHERE source_name = 'Los Angeles International Airport'
      AND source_subsector = 'domestic-aviation'
    ORDER BY operation_pct DESC
    LIMIT 8
""").show()

┌───────────────────────────────────┬─────────────────────────────┬───────────────┐
│            source_name            │       source_operator       │ operation_pct │
│              varchar              │           varchar           │    double     │
├───────────────────────────────────┼─────────────────────────────┼───────────────┤
│ Los Angeles International Airport │ Delta Air Lines             │        25.774 │
│ Los Angeles International Airport │ American Airlines           │        21.478 │
│ Los Angeles International Airport │ United Airlines             │        20.307 │
│ Los Angeles International Airport │ Southwest Airlines          │         7.339 │
│ Los Angeles International Airport │ JetBlue Airways Corporation │          6.72 │
│ Los Angeles International Airport │ Alaska Airlines             │         4.907 │
│ Los Angeles International Airport │ Spirit Airlines             │         3.456 │
│ Los Angeles International Airport │ Hawaiian Airlines           │         

In [26]:
# Test: airports whose airline shares don't add up to 100. No rows means the test passed.
con.sql("""
    SELECT source_id, source_name, source_subsector, ROUND(SUM(operation_pct), 2) AS total_operation_pct
    FROM own
    WHERE source_subsector IN ('domestic-aviation', 'international-aviation')
    GROUP BY ALL
    HAVING ABS(SUM(operation_pct) - 100) > 0.5
""").show()

con.sql("""
    SELECT COUNT(DISTINCT source_operator) AS airlines, COUNT(DISTINCT source_id) AS airport_sources
    FROM own
    WHERE source_subsector IN ('domestic-aviation', 'international-aviation')
""").show()

┌───────────┬─────────────┬──────────────────┬─────────────────────┐
│ source_id │ source_name │ source_subsector │ total_operation_pct │
│  varchar  │   varchar   │     varchar      │       double        │
└───────────┴─────────────┴──────────────────┴─────────────────────┘
                               0 rows                             

┌──────────┬─────────────────┐
│ airlines │ airport_sources │
│  int64   │      int64      │
├──────────┼─────────────────┤
│      201 │             825 │
└──────────┴─────────────────┘



This changes what I wrote in notebook 01. The emissions file only names airports, but the ownership file lists the airlines operating at each airport and their share of its activity. At LAX, Delta has 25.8%, American 21.5% and United 20.3%. The shares add up to 100% at every one of the 825 airport sources, and 201 airlines appear in total.

So aviation can be attributed to airlines after all, using their share of operations rather than ownership. That is a different basis from every other sector, so whether to include it is a decision for ADR 0005, which currently leaves aviation out.

## Summary

**What I learned**

* Ownership joins cleanly to emissions on `source_id` and covers almost all emissions in power, refining, petrochemicals, cement, steel and mining.
* Oil and gas production and transport have no owners, which leaves about 27% of in scope emissions unattributed.
* Most owners have an LEI and a PermID, so company matching can use identifiers.
* The file stores several levels of each ownership chain at once. Summing every row double counts and hands emissions to investors.
* Several subsectors record ownership without percentages.
* Airlines are listed as operators at airports, with shares that add up to 100%.

**Decisions to make**

1. The attribution rule: use the row for the company in our universe and ignore owners above it.
2. What to do when the share is unknown.
3. Whether to bring aviation back into scope using operating shares, which would mean updating ADR 0005.
4. Whether oil and gas production and transport stay in the universe as unattributed emissions or are dropped.

**Checks to build in silver**

1. Every ownership `source_id` exists in the emissions data.
2. Airline operating shares add up to 100% at each airport.
3. Once one owner row is chosen per company, no facility is attributed more than 100%.

**Next:** match owners to SEC filers using LEI and PermID.